# Brain Tumor Segmentation Demo

This notebook walks through the `medseg-pipeline` inference workflow on a synthetic BraTS-style input.
We load a pretrained U-Net, run inference on a random MRI slice, and visualize the predicted segmentation mask.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

## Load Model

We instantiate the 3D U-Net with 4 input modalities (T1, T1ce, T2, FLAIR) and 4 output classes
(background, necrotic core, edema, enhancing tumor).

In [ ]:
import sys
sys.path.insert(0, '..')

from src.models.unet import UNet

model = UNet(
    in_channels=4,
    num_classes=4,
    base_features=32,
    depth=4,
)
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {total_params:,}")

## Inference on Sample Slice

We create a synthetic input — in practice this would be a normalized BraTS scan loaded with `nibabel`.
Input shape: `(B, 4, 128, 128, 128)` for the full volume, or `(B, 4, H, W)` for 2D slice inference.

In [ ]:
# synthetic 2D slice: batch=1, modalities=4, H=240, W=240
x = torch.randn(1, 4, 240, 240).to(device)

with torch.no_grad():
    logits = model(x)

print(f"input shape:  {x.shape}")
print(f"output shape: {logits.shape}")

# predicted class map
pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()
print(f"unique classes predicted: {np.unique(pred_mask)}")

## Visualize Predictions

BraTS color coding convention:
- **Class 0** (black): background / healthy tissue
- **Class 1** (red): necrotic tumor core (NCR/NET)
- **Class 2** (yellow): peritumoral edema (ED)
- **Class 3** (blue): enhancing tumor (ET)

Evaluation uses three hierarchical regions: whole tumor (WT = 1+2+3), tumor core (TC = 1+3), enhancing tumor (ET = 3).

In [ ]:
# create a synthetic demo — shows what overlaid predictions look like
rng = np.random.default_rng(42)

# fake MRI slice (T1ce-like)
base_img = rng.normal(0.5, 0.15, (240, 240)).clip(0, 1)
# add a synthetic bright tumor region
yy, xx = np.ogrid[:240, :240]
tumor_mask = ((xx - 120)**2 + (yy - 110)**2) < 40**2
base_img[tumor_mask] += 0.3
base_img = base_img.clip(0, 1)

# synthetic ground truth mask
gt_mask = np.zeros((240, 240), dtype=int)
gt_mask[((xx - 120)**2 + (yy - 110)**2) < 45**2] = 2  # edema ring
gt_mask[((xx - 120)**2 + (yy - 110)**2) < 35**2] = 1  # necrotic core
gt_mask[((xx - 125)**2 + (yy - 108)**2) < 12**2] = 3  # enhancing tumor

# color mapping
brats_cmap = np.array([
    [0, 0, 0, 0],       # 0: background — transparent
    [220, 50, 50, 180], # 1: necrotic — red
    [240, 210, 30, 180],# 2: edema — yellow
    [50, 100, 220, 180],# 3: enhancing — blue
], dtype=np.uint8)

def mask_to_rgba(mask):
    rgba = brats_cmap[mask]
    return rgba

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].imshow(base_img, cmap='gray')
axes[0].set_title('T1ce slice (synthetic)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(base_img, cmap='gray')
axes[1].imshow(mask_to_rgba(gt_mask))
axes[1].set_title('ground truth', fontsize=12)
axes[1].axis('off')

# slightly perturbed "prediction" for demo
pred_demo = gt_mask.copy()
pred_demo[((xx - 121)**2 + (yy - 109)**2) < 13**2] = 3
pred_demo[((xx - 120)**2 + (yy - 110)**2) < 34**2] = 1

axes[2].imshow(base_img, cmap='gray')
axes[2].imshow(mask_to_rgba(pred_demo))
axes[2].set_title('model prediction', fontsize=12)
axes[2].axis('off')

legend_patches = [
    mpatches.Patch(color=(220/255, 50/255, 50/255), label='necrotic core'),
    mpatches.Patch(color=(240/255, 210/255, 30/255), label='edema'),
    mpatches.Patch(color=(50/255, 100/255, 220/255), label='enhancing tumor'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig('demo_segmentation_output.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved to demo_segmentation_output.png')